# Confidence gating for the IDH hybrid

The parameter-free confidence gate from the paediatric HGG work (RSNA 2026), applied to adult IDH prediction.
Two arms are combined per patient: the **DL arm** (three-member ensemble) and the **ML arm** (handcrafted features + age).

| Strategy | Rule | Fitted on training folds |
|---|---|---|
| A · ML arm | features + age, logistic | coefficients |
| B · DL arm | ensemble logit, recalibrated | intercept and slope |
| C · Fixed blend | a·p_ML + (1-a)·p_DL | a, on inner-CV predictions |
| **D1 · Confidence gate (RSNA)** | w_ML = \|p_ML - 0.5\| / (\|p_ML - 0.5\| + \|p_DL - 0.5\|) | nothing |
| D2 · Gate, threshold-centred | same, with confidence = \|logit p - logit t\| and t each arm's Youden threshold | the two thresholds |
| E · Stacked | logistic on [logit p_ML, logit p_DL] | coefficients |
| H · Full hybrid (paper) | logistic on ensemble logit + 8 features + age | coefficients |

Everything, including every operating threshold, is chosen inside the training folds of a site- and class-stratified
5-fold split repeated 5 times (the same splits as the hybrid notebook). D2 exists because IDH-mutant prevalence is 13%:
the decision boundary sits near 0.15, not 0.5, so "distance from 0.5" mostly measures how wildtype-looking a case is.

**Pre-declared comparisons:** D1 vs the better single arm, D1 vs H, D2 vs D1 (paired bootstrap). Everything else is
exploratory. Runtime: a few minutes on CPU.

## 1. Configuration

In [ ]:
from pathlib import Path
PROJECT = Path.cwd()
MANIFEST = PROJECT / 'outputs' / 'patient_manifest.csv'
FEATURES = PROJECT / 'outputs' / 'idh_imaging_correlates' / 'features.csv'
RADGENOME_OOF = PROJECT / 'outputs' / 'selected_three_reliability_20260926_192650' / 'patient_predictions.csv'
MASK_TABLE = PROJECT / 'outputs' / 'v2_mask_content_audit.csv'      # mask provenance for the subgroup analysis
OUTPUT_DIR = PROJECT / 'outputs' / 'idh_gating'
ASSUME_GRADE4_SITES = {'UPENN'}
FROZEN_ENSEMBLE_THRESHOLD = 0.14600733667612076   # reported for reference only (selected on the same OOF data)

N_FOLDS, REPEATS, N_BOOT, SEED = 5, 5, 2000, 20261021
SENS_TARGET = 0.90
C_REG = 1.0                      # L2 strength for every logistic model (fixed in advance, not tuned)
N_BOOT_FAST = 1000
AGE_MATCH_CALIPER, AGE_MATCH_K = 2.0, 3

## 2. Cohort (identical to the hybrid notebook)

In [ ]:
import json, re, time, warnings, platform
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=RuntimeWarning)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__)

def norm_name(c): return ' ' + re.sub(r'[^a-z0-9]+', ' ', c.lower()) + ' '
def binary(v):
    v = str(v).strip().lower()
    return 1 if v in ('1', '1.0', 'true', 'mutant') else 0 if v in ('0', '0.0', 'false', 'wildtype', 'wt') else np.nan
def grade4(v):
    v = str(v).strip().upper().replace('GRADE', '').replace('WHO', '').replace('CNS', '').strip(' :-')
    if v in ('4', '4.0', 'IV') or 'GLIOBLASTOMA' in v or v == 'GBM': return 1.0
    if v in ('1', '2', '3', '1.0', '2.0', '3.0', 'I', 'II', 'III'): return 0.0
    return np.nan

def metadata_lookup(raw):
    grade, age = {}, {}
    for meta, g in raw.groupby('metadata_path'):
        if not meta or not Path(meta).exists():
            print('  metadata not found:', meta); continue
        md_ = pd.read_csv(meta, sep='\t' if meta.lower().endswith(('.tsv', '.txt')) else ',', dtype=str,
                          keep_default_na=False, encoding_errors='replace')
        to_pid = {}
        for r in g.itertuples():
            for k in {str(getattr(r, 'subject_id', '')), r.patient_id.split(':', 1)[-1]}:
                if k:
                    to_pid[k.strip()] = r.patient_id
                    to_pid.setdefault(re.sub(r'_\d+$', '', k.strip()), r.patient_id)
        idcol = max(md_.columns, key=lambda c: md_[c].str.strip().isin(to_pid).sum())
        gcol = next((c for c in md_.columns if ' grade ' in norm_name(c)), None)
        acol = next((c for c in md_.columns if ' age ' in norm_name(c)
                     and pd.to_numeric(md_[c], errors='coerce').notna().mean() > .5), None)
        for row in md_.to_dict('records'):
            pid = to_pid.get(str(row[idcol]).strip())
            if pid:
                if gcol: grade[pid] = row[gcol]
                if acol: age[pid] = row[acol]
        print(f'  {Path(meta).name}: id {idcol!r}, grade {gcol!r}, age {acol!r}')
    return grade, age

raw = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False)
g_map, a_map = metadata_lookup(raw)
cohort = pd.DataFrame(dict(patient_id=raw.patient_id, site=raw.site.str.upper(), y=raw.idh.map(binary),
                           age=pd.to_numeric(raw.patient_id.map(a_map), errors='coerce'),
                           grade4=raw.patient_id.map(g_map).map(grade4)))
cohort.loc[cohort.grade4.isna() & cohort.site.isin(ASSUME_GRADE4_SITES), 'grade4'] = 1.0
cohort['grade_group'] = cohort.grade4.map({1.0: 'grade 4', 0.0: 'grade 2-3'}).fillna('unknown')
if Path(MASK_TABLE).exists():
    mt = pd.read_csv(MASK_TABLE, dtype=str, keep_default_na=False)[['patient_id', 'source']]
    cohort = cohort.merge(mt.rename(columns={'source': 'mask_source'}), on='patient_id', how='left')
else:
    cohort['mask_source'] = 'unknown'

feats = pd.read_csv(FEATURES)
feats = feats[feats.error.isna() | (feats.error.astype(str).str.len() == 0)].drop(columns=['site'])
oof = pd.read_csv(RADGENOME_OOF)
steps = [('manifest', len(cohort))]
d = cohort[cohort.y.isin([0, 1])]; steps.append(('IDH known', len(d)))
d = d.merge(oof[['patient_id', 'probability', 'fold']], on='patient_id'); steps.append(('with ensemble OOF prediction', len(d)))
d = d.merge(feats, on='patient_id'); steps.append(('with imaging features', len(d)))
d = d[d.age.notna()]; steps.append(('with age', len(d)))
CONV = ['log_vol_whole', 'log_vol_enhancing', 'enhancing_frac_whole', 'no_enhancement', 'necrosis_frac', 'edema_ratio_log']
MISM = ['mismatch_index', 'mismatch_frac']
ADV = ['nadc_whole_median', 'nadc_nonenh_p10', 'nperf_whole_p90']
d['necrosis_frac'] = d.necrosis_frac.fillna(0.0)
d = d.dropna(subset=CONV + MISM).reset_index(drop=True); steps.append(('complete conventional + mismatch features', len(d)))
d['y'] = d.y.astype(int)
p_ = d.probability.clip(1e-6, 1 - 1e-6); d['dl_logit'] = np.log(p_ / (1 - p_))
display(pd.DataFrame(steps, columns=['step', 'patients']))
display(pd.crosstab([d.site, d.grade_group], d.y.map({1: 'IDH-mutant', 0: 'IDH-wildtype'}), margins=True))

In [ ]:
def site_z(frame, cols_):
    out = frame.copy()
    for c in cols_:
        if c == 'no_enhancement': continue
        out[c] = frame.groupby('site')[c].transform(lambda s: (s - s.mean()) / (s.std(ddof=0) + 1e-9))
    return out
Z = site_z(d, CONV + MISM + [c for c in ADV if c in d])

MODELS = {
    'Age': ['age'],
    'Features': CONV + MISM,
    'Features + age': CONV + MISM + ['age'],
    'Ensemble (recalibrated)': ['dl_logit'],
    'Ensemble + age': ['dl_logit', 'age'],
    'Ensemble + features': ['dl_logit'] + CONV + MISM,
    'Hybrid': ['dl_logit'] + CONV + MISM + ['age'],
}
RAW = 'Ensemble (raw)'
ORDER = ['Age', 'Features', 'Features + age', RAW, 'Ensemble (recalibrated)', 'Ensemble + age', 'Ensemble + features', 'Hybrid']

def fit(X, y):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C_REG, max_iter=5000)).fit(X, y)

def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p); k = np.argmax(tpr - fpr); return float(min(thr[k], 1.0))
def sens_threshold(y, p, target=SENS_TARGET):
    fpr, tpr, thr = roc_curve(y, p); ok = np.where(tpr >= target)[0]; return float(min(thr[ok[0]], 1.0))

def inner_thresholds(X, y, strat, seed, raw_p=None):
    # thresholds chosen on the TRAINING fold only, from inner-CV probabilities
    if raw_p is not None:
        return youden_threshold(y, raw_p), sens_threshold(y, raw_p)
    pin = np.zeros(len(y))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, strat):
        pin[te] = fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return youden_threshold(y, pin), sens_threshold(y, pin)

def run_cv(frame, models, repeats=REPEATS, include_raw=True):
    y = frame.y.values; strat = (frame.site + '_' + frame.y.astype(str)).values
    names = list(models) + ([RAW] if include_raw else [])
    P = {m: np.zeros((repeats, len(frame))) for m in names}
    DY = {m: np.zeros((repeats, len(frame)), int) for m in names}
    DS = {m: np.zeros((repeats, len(frame)), int) for m in names}
    TH = {m: [] for m in names}
    for r in range(repeats):
        for k, (tr, te) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + r).split(frame, strat)):
            for m in names:
                if m == RAW:
                    p_te = frame.probability.values[te]
                    ty, ts = inner_thresholds(None, y[tr], None, None, raw_p=frame.probability.values[tr])
                else:
                    X = frame[models[m]].values
                    p_te = fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
                    ty, ts = inner_thresholds(X[tr], y[tr], strat[tr], SEED + 100 * r + k)
                P[m][r, te] = p_te; DY[m][r, te] = p_te >= ty; DS[m][r, te] = p_te >= ts
                TH[m].append((ty, ts))
    out = {}
    for m in names:
        out[m] = dict(p=P[m].mean(0), pred_youden=(DY[m].mean(0) >= .5).astype(int),
                      pred_sens90=(DS[m].mean(0) >= .5).astype(int), thresholds=np.array(TH[m]))
    return out

def calib(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6); lg = np.log(p / (1 - p))
    m = LogisticRegression(penalty=None, max_iter=5000).fit(lg.reshape(-1, 1), y)
    slope = float(m.coef_[0][0])
    # calibration-in-the-large on the log-odds scale: observed minus mean predicted (approximation of the offset model)
    citl = float(np.log(y.mean() / (1 - y.mean())) - np.log(p.mean() / (1 - p.mean())))
    return citl, slope

def ece(y, p, bins=10, adaptive=False):
    if adaptive:
        edges = np.unique(np.quantile(p, np.linspace(0, 1, bins + 1)))
    else:
        edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1], right=True), 0, len(edges) - 2)
    return float(sum(abs(y[idx == b].mean() - p[idx == b].mean()) * (idx == b).mean() for b in np.unique(idx)))

def binary_metrics(y, pred):
    tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    div = lambda a, b: a / b if b else np.nan
    sens, spec = div(tp, tp + fn), div(tn, tn + fp); ppv, npv = div(tp, tp + fp), div(tn, tn + fn)
    f1 = div(2 * tp, 2 * tp + fp + fn); f1n = div(2 * tn, 2 * tn + fp + fn)
    mcc_den = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return dict(tp=tp, fp=fp, tn=tn, fn=fn, sensitivity=sens, specificity=spec, ppv=ppv, npv=npv, f1=f1,
                macro_f1=(f1 + f1n) / 2 if np.isfinite(f1) and np.isfinite(f1n) else np.nan,
                balanced_accuracy=(sens + spec) / 2, mcc=div(tp * tn - fp * fn, mcc_den))

def all_metrics(y, res):
    p = res['p']; citl, slope = calib(y, p)
    out = dict(n=len(y), idh_mutant=int(y.sum()), auroc=roc_auc_score(y, p), average_precision=average_precision_score(y, p),
               brier=float(np.mean((p - y) ** 2)), ece=ece(y, p), ece_adaptive=ece(y, p, adaptive=True),
               calibration_in_the_large=citl, calibration_slope=slope)
    out.update({f'{k}@youden': v for k, v in binary_metrics(y, res['pred_youden']).items()})
    out.update({f'{k}@sens90': v for k, v in binary_metrics(y, res['pred_sens90']).items()})
    return out

KEY = ['auroc', 'average_precision', 'brier', 'ece', 'ece_adaptive', 'calibration_slope',
       'sensitivity@youden', 'specificity@youden', 'ppv@youden', 'npv@youden', 'f1@youden', 'macro_f1@youden',
       'balanced_accuracy@youden', 'mcc@youden', 'sensitivity@sens90', 'specificity@sens90', 'ppv@sens90', 'f1@sens90']

def bootstrap(frame, results, names, reps=N_BOOT, seed=SEED, compare=()):
    rng = np.random.default_rng(seed); y = frame.y.values
    groups = [np.where(((frame.site + '_' + frame.y.astype(str)) == g).values)[0]
              for g in sorted(set(frame.site + '_' + frame.y.astype(str)))]
    samples = {m: {k: [] for k in KEY} for m in names}
    diffs = {(a, b): {k: [] for k in KEY} for a, b in compare}
    for _ in range(reps):
        idx = np.concatenate([rng.choice(g, len(g)) for g in groups])
        cache = {}
        for m in set(names) | {x for pair in compare for x in pair}:
            r = results[m]
            cache[m] = all_metrics(y[idx], dict(p=r['p'][idx], pred_youden=r['pred_youden'][idx], pred_sens90=r['pred_sens90'][idx]))
        for m in names:
            for k in KEY: samples[m][k].append(cache[m][k])
        for a, b in compare:
            for k in KEY: diffs[(a, b)][k].append(cache[a][k] - cache[b][k])
    def interval(k, v, pt):
        q = np.nanpercentile(v, [2.5, 97.5])
        if k.startswith('ece'):        # ECE is biased upward under resampling, so bootstrap intervals are not valid: report the point only
            return np.array([np.nan, np.nan])
        return q
    full = {m: all_metrics(y, results[m]) for m in names}
    ci = {m: {k: interval(k, v, full[m][k]) for k, v in samples[m].items()} for m in names}
    dci = {pair: {k: (np.nanmean(v), *np.nanpercentile(v, [2.5, 97.5])) for k, v in dd.items()} for pair, dd in diffs.items()}
    return ci, dci

def fmt_ci(v, ci): return f'{v:.3f}' if np.isnan(ci[0]) else f'{v:.3f} ({ci[0]:.3f}-{ci[1]:.3f})'

## 3. Nested evaluation of all strategies

For each outer training fold: inner 5-fold CV gives out-of-fold arm probabilities for the training patients; on those
we choose the blend weight a, fit the stacked model and set every strategy's Youden and 90%-sensitivity thresholds.
The arms are then refitted on the whole training fold and applied to the test fold.

In [ ]:
from scipy.stats import rankdata
warnings.filterwarnings('ignore', category=FutureWarning)
ML_COLS, DL_COLS = CONV + MISM + ['age'], ['dl_logit']
STRATS = ['A · ML arm (features + age)', 'B · DL arm (ensemble)', 'C · Fixed blend', 'D1 · Confidence gate (RSNA)',
          'D2 · Gate, threshold-centred', 'E · Stacked', 'H · Full hybrid']
A_, B_, C_, D1_, D2_, E_, H_ = STRATS
def lg(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def gate(pm, pd_, centre_m=0.5, centre_d=0.5, logodds=False):
    if logodds: cm, cd = np.abs(lg(pm) - lg(centre_m)), np.abs(lg(pd_) - lg(centre_d))
    else:       cm, cd = np.abs(pm - centre_m), np.abs(pd_ - centre_d)
    w = np.where(cm + cd > 0, cm / np.maximum(cm + cd, 1e-12), 0.5)
    return w * pm + (1 - w) * pd_, w
def youden_t(y, p): return youden_threshold(y, p)
def sens_t(y, p):   return sens_threshold(y, p)
ALPHAS = np.round(np.linspace(0, 1, 21), 2)

def fit_combiners(y, pm, pd_):
    """Everything a strategy needs, from out-of-fold arm probabilities of TRAINING patients only."""
    a = ALPHAS[int(np.argmax([roc_auc_score(y, al * pm + (1 - al) * pd_) for al in ALPHAS]))]
    stk = LogisticRegression(C=C_REG, max_iter=5000).fit(np.c_[lg(pm), lg(pd_)], y)
    tm, td = youden_t(y, pm), youden_t(y, pd_)
    return dict(a=a, stk=stk, tm=tm, td=td)

def combine(pm, pd_, f):
    out = {A_: pm, B_: pd_, C_: f['a'] * pm + (1 - f['a']) * pd_}
    out[D1_], w1 = gate(pm, pd_)
    out[D2_], w2 = gate(pm, pd_, f['tm'], f['td'], logodds=True)
    out[E_] = f['stk'].predict_proba(np.c_[lg(pm), lg(pd_)])[:, 1]
    return out, w1, w2

y = Z.y.values; strat = (Z.site + '_' + Z.y.astype(str)).values
XM, XD, XH = Z[ML_COLS].values, Z[DL_COLS].values, Z[MODELS['Hybrid']].values
n = len(Z); P = {s: np.zeros((REPEATS, n)) for s in STRATS}
DY = {s: np.zeros((REPEATS, n), int) for s in STRATS}; DS = {s: np.zeros((REPEATS, n), int) for s in STRATS}
W1 = np.zeros((REPEATS, n)); W2 = np.zeros((REPEATS, n)); ALPHA_USED = []
t0 = time.time()
for r in range(REPEATS):
    for k, (tr, te) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + r).split(Z, strat)):
        # inner out-of-fold arm probabilities for the training patients
        pm_in, pd_in, ph_in = np.zeros(len(tr)), np.zeros(len(tr)), np.zeros(len(tr))
        for itr, ite in StratifiedKFold(5, shuffle=True, random_state=SEED + 100 * r + k).split(tr, strat[tr]):
            a_, b_ = tr[itr], tr[ite]
            pm_in[ite] = fit(XM[a_], y[a_]).predict_proba(XM[b_])[:, 1]
            pd_in[ite] = fit(XD[a_], y[a_]).predict_proba(XD[b_])[:, 1]
            ph_in[ite] = fit(XH[a_], y[a_]).predict_proba(XH[b_])[:, 1]
        f = fit_combiners(y[tr], pm_in, pd_in); ALPHA_USED.append(f['a'])
        train_p, _, _ = combine(pm_in, pd_in, f); train_p[H_] = ph_in
        # final arms on the whole training fold, applied to the test fold
        pm, pd_ = fit(XM[tr], y[tr]).predict_proba(XM[te])[:, 1], fit(XD[tr], y[tr]).predict_proba(XD[te])[:, 1]
        test_p, w1, w2 = combine(pm, pd_, f); test_p[H_] = fit(XH[tr], y[tr]).predict_proba(XH[te])[:, 1]
        W1[r, te], W2[r, te] = w1, w2
        for s in STRATS:
            ty, ts = youden_t(y[tr], train_p[s]), sens_t(y[tr], train_p[s])
            P[s][r, te] = test_p[s]; DY[s][r, te] = test_p[s] >= ty; DS[s][r, te] = test_p[s] >= ts
cvg = {s: dict(p=P[s].mean(0), pred_youden=(DY[s].mean(0) >= .5).astype(int), pred_sens90=(DS[s].mean(0) >= .5).astype(int))
       for s in STRATS}
w_ml1, w_ml2 = W1.mean(0), W2.mean(0)
print(f'{REPEATS}x{N_FOLDS} nested CV in {time.time() - t0:.0f}s | blend weight a on ML arm: median {np.median(ALPHA_USED):.2f} '
      f'(range {min(ALPHA_USED):.2f}-{max(ALPHA_USED):.2f})')
pd.DataFrame({'patient_id': Z.patient_id, 'site': Z.site, 'y': y, 'age': Z.age, 'grade_group': Z.grade_group,
              'w_ML_gate': w_ml1, 'w_ML_gate_threshold_centred': w_ml2,
              **{f'p[{s}]': cvg[s]['p'] for s in STRATS}, **{f'call[{s}]': cvg[s]['pred_youden'] for s in STRATS}}
             ).to_csv(OUTPUT_DIR / 'gating_cv_predictions.csv', index=False)

## 4. Performance

In [ ]:
point = {s: all_metrics(y, cvg[s]) for s in STRATS}
best_arm = A_ if point[A_]['auroc'] >= point[B_]['auroc'] else B_
COMPARE = [(D1_, best_arm), (D1_, H_), (D2_, D1_), (D1_, B_), (D2_, H_), (E_, H_), (C_, D1_)]
ci, dci = bootstrap(Z, cvg, STRATS, compare=COMPARE)
SHOW = [('auroc', 'AUROC'), ('average_precision', 'AP'), ('brier', 'Brier'), ('ece', 'ECE'), ('calibration_slope', 'Slope'),
        ('sensitivity@youden', 'Sens.'), ('specificity@youden', 'Spec.'), ('ppv@youden', 'PPV'), ('f1@youden', 'F1'),
        ('mcc@youden', 'MCC'), ('specificity@sens90', 'Spec. @90% sens.')]
G1 = pd.DataFrame([{'strategy': s, **{lab: fmt_ci(point[s][k], ci[s][k]) for k, lab in SHOW}} for s in STRATS])
display(G1); G1.to_csv(OUTPUT_DIR / 'gating_metrics.csv', index=False)
pd.DataFrame([{'strategy': s, **point[s]} for s in STRATS]).to_csv(OUTPUT_DIR / 'gating_metrics_raw.csv', index=False)
rows = []
for (a, b) in COMPARE:
    d_ = dci[(a, b)]
    rows.append({'comparison': f'{a}  minus  {b}', 'pre-declared': (a, b) in [(D1_, best_arm), (D1_, H_), (D2_, D1_)],
                 **{lab: f'{d_[k][0]:+.3f} ({d_[k][1]:+.3f} to {d_[k][2]:+.3f})' for k, lab in
                    [('auroc', 'AUROC'), ('average_precision', 'AP'), ('brier', 'Brier'), ('specificity@youden', 'Spec.'), ('f1@youden', 'F1')]}})
G2 = pd.DataFrame(rows); display(G2); G2.to_csv(OUTPUT_DIR / 'gating_paired_differences.csv', index=False)
print('Better single arm:', best_arm)

## 5. How the gate behaves

Left: each patient's arm confidences, with the ML arm's errors marked (the RSNA figure). Right: the per-patient weight
the gate gives the ML arm. Then: who the gate hands to which arm, by grade, age and enhancement.

In [ ]:
pA, pB = cvg[A_]['p'], cvg[B_]['p']
errA = cvg[A_]['pred_youden'] != y; errB = cvg[B_]['pred_youden'] != y
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
ax = axes[0]
cA, cB = np.abs(pA - .5), np.abs(pB - .5)
ax.scatter(cA[~errA], cB[~errA], s=6, color='#9a9a96', alpha=.5, label='ML arm correct')
ax.scatter(cA[errA], cB[errA], s=22, marker='x', color='#c0392b', label='ML arm wrong')
ax.plot([0, .5], [0, .5], color='k', lw=.7); ax.set_xlabel('ML-arm confidence |p - 0.5|'); ax.set_ylabel('DL-arm confidence |p - 0.5|')
ax.set_title('D1 gate: arm confidences'); ax.legend(frameon=False, fontsize=8)
ax = axes[1]
tmA, tmB = youden_t(y, pA), youden_t(y, pB)
lcA, lcB = np.abs(lg(pA) - lg(tmA)), np.abs(lg(pB) - lg(tmB))
ax.scatter(lcA[~errA], lcB[~errA], s=6, color='#9a9a96', alpha=.5, label='ML arm correct')
ax.scatter(lcA[errA], lcB[errA], s=22, marker='x', color='#c0392b', label='ML arm wrong')
m_ = max(lcA.max(), lcB.max()); ax.plot([0, m_], [0, m_], color='k', lw=.7)
ax.set_xlabel('ML-arm confidence |logit p - logit t|'); ax.set_ylabel('DL-arm confidence'); ax.set_title('D2 gate: threshold-centred')
ax = axes[2]
ax.hist(w_ml1, bins=30, alpha=.6, color='#3b8e3f', label='D1 (centre 0.5)')
ax.hist(w_ml2, bins=30, alpha=.6, color='#2a6fdb', label='D2 (threshold-centred)')
ax.set_xlabel('weight on ML arm (features + age)'); ax.set_ylabel('patients'); ax.legend(frameon=False, fontsize=8)
ax.set_title('Per-patient gate weights')
for a in axes: a.spines[['top', 'right']].set_visible(False)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'gate_behaviour.png', dpi=200); plt.show()

band = pd.cut(Z.age, [0, 40, 55, 200], right=False, labels=['<40', '40-54', '>=55']).astype(str)
enh = np.where(Z.no_enhancement.values > .5, 'non-enhancing', 'enhancing')
rows = []
for var, vals in [('all', np.array(['all'] * n)), ('grade', Z.grade_group.values), ('age', band.values), ('enhancement', enh)]:
    for lev in pd.unique(vals):
        k = vals == lev
        rows.append({'group': f'{var}: {lev}', 'n': int(k.sum()), 'mutant': int(y[k].sum()),
                     'D1 mean w_ML': round(w_ml1[k].mean(), 2), 'D1 % patients led by DL (w_ML<0.5)': round(100 * (w_ml1[k] < .5).mean(), 1),
                     'D2 mean w_ML': round(w_ml2[k].mean(), 2), 'D2 % led by DL': round(100 * (w_ml2[k] < .5).mean(), 1),
                     'ML-arm errors / DL-arm errors': f'{int(errA[k].sum())} / {int(errB[k].sum())}'})
G3 = pd.DataFrame(rows); display(G3); G3.to_csv(OUTPUT_DIR / 'gate_weights_by_group.csv', index=False)
both = errA & errB
print(f'Errors: ML arm {errA.sum()}, DL arm {errB.sum()}, both {both.sum()} (a gate cannot fix cases where both arms are wrong).')
print(f'ML-arm errors where the DL arm was more confident (D2 scale): {int((errA & (lcB > lcA)).sum())} of {int(errA.sum())}.')

## 6. Subgroups, age matching and deferral

Does the gate keep the ensemble's advantages (grade 2-3, under 40) and features+age's (grade 4, non-enhancing)?
Does it hold up when age is matched out? And does deferral by the gate's own confidence work as well as the hybrid's?

In [ ]:
SG_STRATS = [A_, B_, D1_, D2_, E_, H_]
rows = []
for var, vals in [('grade', Z.grade_group.values), ('age', band.values), ('enhancement', enh), ('site', Z.site.values)]:
    for lev in sorted(pd.unique(vals)):
        k = vals == lev
        if y[k].sum() < 5 or (1 - y[k]).sum() < 5: continue
        rows.append({'group': f'{var}: {lev}', 'n': int(k.sum()), 'mutant': int(y[k].sum()),
                     **{s: round(roc_auc_score(y[k], cvg[s]['p'][k]), 3) for s in SG_STRATS}})
G4 = pd.DataFrame(rows); display(G4); G4.to_csv(OUTPUT_DIR / 'gating_subgroups.csv', index=False)

# age-matched sample: same procedure as the novelty notebook
rng = np.random.default_rng(SEED); used = set(); sets = []; AGE = Z.age.values; sites = Z.site.values
mut = np.where(y == 1)[0].copy(); rng.shuffle(mut)
for m in mut:
    cand = np.where((y == 0) & (sites == sites[m]) & (np.abs(AGE - AGE[m]) <= AGE_MATCH_CALIPER))[0]
    cand = sorted((c for c in cand if c not in used), key=lambda c: (abs(AGE[c] - AGE[m]), c))[:AGE_MATCH_K]
    if cand: used.update(cand); sets.append(np.array([m] + list(cand)))
MATCHED = np.concatenate(sets)
def clus_ci(s):
    rng = np.random.default_rng(SEED); v = []
    for _ in range(N_BOOT_FAST):
        i = np.concatenate([sets[j] for j in rng.integers(0, len(sets), len(sets))]); v.append(roc_auc_score(y[i], cvg[s]['p'][i]))
    return np.percentile(v, [2.5, 97.5])
G5 = pd.DataFrame([{'strategy': s, 'AUROC, all': round(point[s]['auroc'], 3),
                    'AUROC, age-matched': fmt_ci(roc_auc_score(y[MATCHED], cvg[s]['p'][MATCHED]), clus_ci(s))} for s in STRATS])
print(f'Age-matched sample: {len(sets)} mutants, {len(MATCHED) - len(sets)} wildtype')
display(G5); G5.to_csv(OUTPUT_DIR / 'gating_age_matched.csv', index=False)

# deferral by each strategy's own confidence: distance of its probability from its (median nested) Youden threshold
COV = np.round(np.arange(.5, 1.0001, .01), 2)
def curve(p, call, thr):
    err = (call != y).astype(float); order = np.argsort(np.abs(lg(p) - lg(thr)))[::-1]   # most confident first
    cum = np.cumsum(err[order]); nk = np.maximum(np.round(COV * n).astype(int), 1); return cum[nk - 1] / nk
rows = []; fig, ax = plt.subplots(figsize=(5.2, 3.8))
for s, c in [(H_, '#2a6fdb'), (D1_, '#7b1fa2'), (D2_, '#d98a1c'), (B_, '#5b5b58')]:
    thr = youden_t(y, cvg[s]['p'])   # display only: the ranking uses distance from a single threshold
    cu = curve(cvg[s]['p'], cvg[s]['pred_youden'], thr); ax.plot(COV * 100, cu * 100, color=c, lw=1.6, label=s)
    k80 = int(np.where(COV == .8)[0][0])
    rows.append({'strategy': s, 'error rate, no deferral': round(1 - (cvg[s]['pred_youden'] == y).mean(), 3),
                 'error rate, 20% deferred': round(cu[k80], 3), 'AURC (50-100% coverage)': round(cu.mean(), 4)})
ax.invert_xaxis(); ax.set_xlabel('patients kept (%)'); ax.set_ylabel('error rate among kept (%)'); ax.legend(frameon=False, fontsize=7)
ax.spines[['top', 'right']].set_visible(False); fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'gating_risk_coverage.png', dpi=200); plt.show()
G6 = pd.DataFrame(rows); display(G6); G6.to_csv(OUTPUT_DIR / 'gating_deferral.csv', index=False)
print('Deferral ranks patients by distance from a single full-data Youden threshold for display; the error calls are the nested ones.')

## Report

In [ ]:
def mdt(df):
    cols = list(df.columns)
    return '\n'.join(['| ' + ' | '.join(map(str, cols)) + ' |', '|' + '---|' * len(cols)] +
                     ['| ' + ' | '.join(str(v) for v in r) + ' |' for r in df.values])
lines = ['# Confidence gating for the IDH hybrid', '', f'Run {time.strftime("%Y-%m-%d %H:%M")} | {n} patients, {int(y.sum())} IDH-mutant | '
         f'{REPEATS}x{N_FOLDS} nested CV. DL arm = ensemble; ML arm = features + age. Better single arm: {best_arm}.', '',
         '## Performance (95% CI)', '', mdt(G1), '', '## Paired differences (95% CI)', '', mdt(G2), '',
         '## Gate weights by group', '', mdt(G3), '', '## Subgroup AUROC', '', mdt(G4), '',
         '## Age-matched AUROC', '', mdt(G5), '', '## Deferral by each strategy\'s own confidence', '', mdt(G6), '',
         '## Notes', '', '- Pre-declared: D1 vs the better single arm, D1 vs H, D2 vs D1. Everything else is exploratory.',
         '- D1 is exactly the RSNA rule (distance from 0.5). D2 measures distance from each arm\'s training-fold Youden threshold on the log-odds scale.',
         '- The fixed blend weight and the stacked model are fitted on inner-CV predictions of the training folds only.',
         '- ECE intervals are not reported (biased under resampling).']
(OUTPUT_DIR / 'REPORT.md').write_text('\n'.join(lines), encoding='utf-8'); print('Wrote', OUTPUT_DIR / 'REPORT.md')